# RAGAS 구현하기
기존 RAG 과정에 ReverseHyDE를 도입 → 어느 정도의 성능 향상이 있는지 RAGAS 평가지표를 기준으로 확인해보고자 했다.

In [1]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters

In [4]:
import os

In [5]:
# 2. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [6]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 32.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 39.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
from langchain_openai import ChatOpenAI

In [10]:
# 8. OpenAI LLM 설정 (GPT-4o 또는 GPT-3.5-turbo 등)
llm_openai = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

## 토크나이저 정의 및 PDF 문서 로드

In [9]:
!pip install tiktoken

In [11]:
import tiktoken

In [12]:
# 5. 토크나이저 설정
tokenizer = tiktoken.get_encoding("cl100k_base")   # GPT-4, GPT-3.5-turbo 모델들이 사용하는 인코딩 방식

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

In [12]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [13]:
!pip install langchain-chroma

In [13]:
from langchain_chroma import Chroma

In [14]:
from langchain_community.document_loaders import PyPDFLoader

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

## Text Splitting

In [16]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/drive/MyDrive/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50, length_function = tiktoken_len)
texts = text_splitter.split_documents(pages)

In [17]:
tiktoken_length = []
for doc in texts:
    # chunk(Document 객체)가 아니라 내부의 텍스트(.page_content)를 전달
    tiktoken_length.append(tiktoken_len(doc.page_content))

print(f"생성된 조각 개수: {len(texts)}")
print(f"조각별 토큰 길이: {tiktoken_length}")

생성된 조각 개수: 182
조각별 토큰 길이: [16, 36, 327, 426, 303, 414, 416, 427, 418, 405, 388, 399, 423, 409, 411, 396, 452, 425, 433, 432, 431, 423, 425, 387, 427, 283, 318, 418, 399, 425, 419, 440, 399, 403, 421, 407, 395, 388, 399, 400, 400, 397, 417, 359, 400, 400, 406, 448, 282, 301, 414, 422, 442, 421, 411, 415, 414, 396, 427, 440, 414, 362, 418, 439, 437, 389, 422, 383, 393, 411, 417, 332, 313, 422, 402, 432, 415, 429, 449, 418, 424, 441, 416, 411, 403, 433, 418, 414, 427, 444, 375, 413, 410, 431, 441, 431, 268, 315, 416, 415, 399, 423, 456, 412, 452, 411, 435, 396, 406, 386, 419, 438, 421, 401, 435, 429, 395, 323, 432, 423, 442, 386, 396, 416, 391, 426, 427, 414, 419, 401, 355, 424, 397, 385, 389, 424, 418, 421, 447, 422, 421, 153, 312, 421, 391, 402, 376, 407, 400, 398, 423, 428, 415, 411, 401, 410, 401, 424, 402, 387, 409, 387, 429, 419, 411, 403, 420, 403, 339, 383, 293, 326, 421, 365, 389, 400, 378, 405, 431, 408, 407, 230]


### 텍스트 임베딩

In [18]:
import openai

In [19]:
client = openai.OpenAI()

In [20]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-3-small
text-embedding-3-large
text-embedding-ada-002


In [21]:
from langchain_openai import OpenAIEmbeddings

## 벡터 데이터베이스(Vector DB) 구축

In [22]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [23]:
db = Chroma.from_documents(texts, embedding_model)

In [24]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [25]:
print(docs[1].page_content)

DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face somehow expressed my life, it was my inner self, 
my fate or my daimon. That was how

## Reverse HyDE 구현

In [26]:
# [핵심 로직] Reverse HyDE 구현 예시
from langchain_core.documents import Document

# 1. 데이터를 담을 빈 리스트 생성
reverse_hyde_texts = []

# 2. 진행 상황 출력
print(f"총 {len(texts)}개의 문서 조각에서 예상 질문 생성을 시작합니다...")

# 3. 루프 실행
for i, doc in enumerate(texts):
    # 진행 상황 출력
    print(f"[{i+1}/{len(texts)}] 번째 조각 처리 중...")

    # LLM에게 예상 질문 생성 요청 (문구는 필요에 따라 조정 가능)
    # response.content는 LLM이 생성한 답변 텍스트입니다.
    response = llm_openai.invoke(f"다음 내용을 읽고, 이 내용이 정답이 될 수 있는 질문 3개만 써줘:\n\n{doc.page_content}")

    # 원본 내용 + 예상 질문 합치기 (이것이 Reverse HyDE의 핵심입니다)
    enhanced_content = f"예상 질문들:\n{response.content}\n\n원본 내용:\n{doc.page_content}"

    # 새로운 Document 객체 생성 (기존 metadata 유지)
    new_doc = Document(page_content=enhanced_content, metadata=doc.metadata)
    reverse_hyde_texts.append(new_doc)

# 4. 강화된 데이터를 벡터 DB에 저장 (루프가 완전히 끝난 후 실행)
# embedding_model은 이미 정의되어 있다고 가정합니다.
docsearch = Chroma.from_documents(
    documents=reverse_hyde_texts,
    embedding=embedding_model,
    collection_name="reverse_hyde_collection" # 선택 사항: 컬렉션 이름 지정
)

print(f"총 {len(reverse_hyde_texts)}개의 강화된 문서로 Vector Database 구축이 완료되었습니다!")

총 182개의 문서 조각에서 예상 질문 생성을 시작합니다...
[1/182] 번째 조각 처리 중...
[2/182] 번째 조각 처리 중...
[3/182] 번째 조각 처리 중...
[4/182] 번째 조각 처리 중...
[5/182] 번째 조각 처리 중...
[6/182] 번째 조각 처리 중...
[7/182] 번째 조각 처리 중...
[8/182] 번째 조각 처리 중...
[9/182] 번째 조각 처리 중...
[10/182] 번째 조각 처리 중...
[11/182] 번째 조각 처리 중...
[12/182] 번째 조각 처리 중...
[13/182] 번째 조각 처리 중...
[14/182] 번째 조각 처리 중...
[15/182] 번째 조각 처리 중...
[16/182] 번째 조각 처리 중...
[17/182] 번째 조각 처리 중...
[18/182] 번째 조각 처리 중...
[19/182] 번째 조각 처리 중...
[20/182] 번째 조각 처리 중...
[21/182] 번째 조각 처리 중...
[22/182] 번째 조각 처리 중...
[23/182] 번째 조각 처리 중...
[24/182] 번째 조각 처리 중...
[25/182] 번째 조각 처리 중...
[26/182] 번째 조각 처리 중...
[27/182] 번째 조각 처리 중...
[28/182] 번째 조각 처리 중...
[29/182] 번째 조각 처리 중...
[30/182] 번째 조각 처리 중...
[31/182] 번째 조각 처리 중...
[32/182] 번째 조각 처리 중...
[33/182] 번째 조각 처리 중...
[34/182] 번째 조각 처리 중...
[35/182] 번째 조각 처리 중...
[36/182] 번째 조각 처리 중...
[37/182] 번째 조각 처리 중...
[38/182] 번째 조각 처리 중...
[39/182] 번째 조각 처리 중...
[40/182] 번째 조각 처리 중...
[41/182] 번째 조각 처리 중...
[42/182] 번째 조각 처리 중...
[43/182]

## Retrieval

In [27]:
!pip install -U langchain langchain-classic

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 362, in run
    resolver = self.make_resolver(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 177, in make_resolver
    return pip._internal.resolution.resolvelib.resolver.Resolver(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 58, in __init__
    self.factory = Factory(
                   ^^^^^^^^
  File "/usr/local/lib/py

In [28]:
# from langchain.chains import RetrievalQA에서 바꾼 것
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [29]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

In [30]:
qa = RetrievalQA.from_chain_type(
    llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(
      search_type="mmr",
      search_kwargs={"k": 3, "fetch_k" : 10}),
    return_source_documents=True)

In [31]:
# 10. 질문 실행 및 결과 출력
query = "Who is Sinclair?"
result = qa.invoke(query)

from IPython.display import Markdown, display
display(Markdown(result["result"]))

Sinclair is the protagonist of Hermann Hesse's novel "Demian." The story follows his journey of self-discovery and personal growth, influenced by his interactions with characters like Max Demian and Frau Eva.

Sinclair is the protagonist of Hermann Hesse's novel "Demian." The story follows his journey of self-discovery and personal growth, influenced by his interactions with characters like Max Demian and Frau Eva.

#### RAG를 사용하지 않은 llm 호출

In [ ]:
# llm2 = ChatOpenAI(
#     model="gpt-4o-mini")
# request = llm2.invoke("how demian looks like")
# display(Markdown(request.content))

## RAGAS

In [29]:
!pip install ragas datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 21.3 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.13.0
    Uninstalling jiter-0.13.0:
      Successfully uninstalled jiter-0.13.0


In [32]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)
import pandas as pd

/tmp/ipykernel_6300/2706617862.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_6300/2706617862.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_6300/2706617862.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_6300/2706617862.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in

In [33]:
# 1. 평가용 질문 및 정답 정의
eval_questions = [
    "싱클레어는 누구니?",
    "데미안이 싱클레어에게 카인과 아벨에 대해 설명한 내용은 무엇인가요?",
    "크로머는 어떤 방식으로 싱클레어를 협박했나요?"
]

ground_truths = [
    "싱클레어는 소설 '데미안'의 주인공으로, 밝은 세계와 어두운 세계 사이에서 방황하며 자아를 찾아가는 인물입니다.",
    "데미안은 카인의 표식이 악의 상징이 아니라, 남들보다 강하고 지적인 사람들을 구별해주는 고귀한 표식이라고 설명했습니다.",
    "크로머는 싱클레어가 이웃집 과수원에서 사과를 훔쳤다는 거짓 고백을 약점 잡아 돈을 요구하며 그를 괴롭혔습니다."
]

In [34]:
# 2. 결과 수집
results = []
for query in eval_questions:
    # QA 시스템 호출
    response = qa.invoke(query)

    # 답변 추출 (RetrievalQA 결과 구조에 따라 수정)
    answer = response.get("result") or response.get("answer")

    # 검색된 근거 추출 (위에서 만든 docsearch 활용)
    retrieved_docs = docsearch.as_retriever().invoke(query)
    context_list = [doc.page_content for doc in retrieved_docs]

    results.append({
        "question": query,
        "answer": answer,
        "contexts": context_list
    })


싱클레어는 헤르만 헤세의 소설 "데미안"의 주인공입니다. 그는 소설에서 자신의 정체성을 찾고 성장하는 과정을 겪으며, 데미안이라는 인물과의 만남을 통해 내면의 갈등과 자기 발견을 경험하게 됩니다. 싱클레어는 자신의 과거와 현재를 대조하며, 그로 인해 다양한 감정을 느끼고 자기혐오와 같은 복잡한 감정도 겪게 됩니다.죄송하지만, 데미안이 싱클레어에게 카인과 아벨에 대해 설명한 구체적인 내용은 제공된 문맥에서 찾을 수 없습니다.크로머는 싱클레어에게 불가능한 요구를 하며 그를 겁주고 굴욕감을 주는 방식으로 협박했습니다. 그런 다음 점차적으로 요구를 완화하면서 싱클레어가 돈이나 선물로 자신을 구제하도록 만들었습니다. 이번에는 싱클레어의 여동생을 산책에 데리고 나오라는 요청을 했고, 싱클레어는 그러한 요구에 절대 동참하지 않겠다고 결심했습니다. 그러나 크로머가 어떻게 보복할지에 대한 두려움도 느꼈습니다.

In [35]:
# 3. 데이터셋 변환
data = {
    "question": [r["question"] for r in results],
    "answer": [r["answer"] for r in results],
    "contexts": [r["contexts"] for r in results],
    "ground_truth": ground_truths
}
dataset = Dataset.from_dict(data)

In [36]:
# 4. Ragas 평가 실행 (OpenAI API 키가 설정되어 있어야 합니다)
print("Ragas 평가를 시작합니다...")
result = evaluate(
    dataset=dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
)

Ragas 평가를 시작합니다...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[3]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[7]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[11]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')


In [37]:
# 5. 결과 확인
print("\n[Ragas 평가 결과]")
print(result)
df = result.to_pandas()
display(df)


[Ragas 평가 결과]
{'context_precision': 0.4444, 'context_recall': 1.0000, 'faithfulness': 0.6111, 'answer_relevancy': nan}


,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,싱클레어는 누구니?,"[예상 질문들:\n1. 이 인물은 자신의 과거와 현재를 어떻게 대조하고 있으며, 그...","싱클레어는 헤르만 헤세의 소설 ""데미안""의 주인공입니다. 그는 소설에서 자신의 정체...","싱클레어는 소설 '데미안'의 주인공으로, 밝은 세계와 어두운 세계 사이에서 방황하며...",0.000000,1.0,0.833333,NaN
1,데미안이 싱클레어에게 카인과 아벨에 대해 설명한 내용은 무엇인가요?,[예상 질문들:\n1. 데미안이 자신의 의지를 통해 다른 사람에게서 원하는 것을 얻...,"죄송하지만, 데미안이 싱클레어에게 카인과 아벨에 대해 설명한 구체적인 내용은 제공된...","데미안은 카인의 표식이 악의 상징이 아니라, 남들보다 강하고 지적인 사람들을 구별해...",0.333333,1.0,0.000000,NaN
2,크로머는 어떤 방식으로 싱클레어를 협박했나요?,"[예상 질문들:\n1. 어떤 요청이 Kromer에 의해 제기되었고, 그 요청에 대한...",크로머는 싱클레어에게 불가능한 요구를 하며 그를 겁주고 굴욕감을 주는 방식으로 협박...,크로머는 싱클레어가 이웃집 과수원에서 사과를 훔쳤다는 거짓 고백을 약점 잡아 돈을 ...,1.000000,1.0,1.000000,NaN


## 정성 평가

In [38]:
# 1. 분석할 질문 리스트 (평가에 사용했던 질문과 동일하게 설정)
analysis_questions = [
    "싱클레어는 누구니?",
    "데미안이 싱클레어에게 카인과 아벨에 대해 설명한 내용은 무엇인가요?",
    "크로머는 어떤 방식으로 싱클레어를 협박했나요?"
]

print("=== Retrieval Analysis (검색 결과 적절성 검토) ===\n")

analysis_results = []

for i, query in enumerate(analysis_questions):
    print(f"🔎 질문 {i+1}: {query}")

    # docsearch(Chroma DB)에서 관련 문서 검색 (k=3)
    # Reverse HyDE로 구축된 DB이므로 '예상 질문'이 포함된 문서를 가져옵니다.
    retrieved_docs = docsearch.as_retriever(search_kwargs={"k": 3}).invoke(query)

    print(f"✅ 검색된 문서 개수: {len(retrieved_docs)}")

    for j, doc in enumerate(retrieved_docs):
        # 검색된 내용의 앞부분 200자만 추출하여 출력
        content_preview = doc.page_content.replace('\n', ' ')[:200]
        source = doc.metadata.get('source', '알 수 없음')
        page = doc.metadata.get('page', '-')

        print(f"   [문서 {j+1}] (출처: {source}, {page}페이지)")
        print(f"   내용 요약: {content_preview}...")
        print("-" * 50)

    print("\n" + "="*80 + "\n")

# (선택 사항) 결과를 표 형태로 정리해서 보고 싶을 때
analysis_data = []
for query in analysis_questions:
    docs = docsearch.as_retriever(search_kwargs={"k": 3}).invoke(query)
    analysis_data.append({
        "Question": query,
        "Top_1_Context": docs[0].page_content[:300] if docs else "검색 결과 없음",
        "Top_2_Context": docs[1].page_content[:300] if len(docs) > 1 else "-",
        "Top_3_Context": docs[2].page_content[:300] if len(docs) > 2 else "-"
    })

df_analysis = pd.DataFrame(analysis_data)
display(df_analysis)

=== Retrieval Analysis (검색 결과 적절성 검토) ===

🔎 질문 1: 싱클레어는 누구니?
✅ 검색된 문서 개수: 3
   [문서 1] (출처: /content/drive/MyDrive/Demian.pdf, 78페이지)
   내용 요약: 예상 질문들: 1. 이 인물은 자신의 과거와 현재를 어떻게 대조하고 있으며, 그로 인해 어떤 감정을 느끼고 있나요? 2. 비트리스가 느끼는 자기혐오의 원인은 무엇이며, 그가 자신을 "아웃캐스트"로 묘사하는 이유는 무엇인가요? 3. 비트리스가 경험한 사랑과 애정의 기억은 현재의 그의 정체성과 어떻게 연결되어 있나요?  원본 내용: BEATRICE  schoo...
--------------------------------------------------
   [문서 2] (출처: /content/drive/MyDrive/Demian.pdf, 88페이지)
   내용 요약: 예상 질문들: 1. 이 그림이 왜 비아트리스의 삶에 큰 영향을 미쳤는가? 2. 비아트리스가 그림을 보고 느낀 감정은 무엇인가? 3. 비아트리스가 꿈에서 그림을 어떻게 인식하게 되었는가?  원본 내용: BEATRICE  This picture haunted my thoughts for a long time and  divided up my life. I ke...
--------------------------------------------------
   [문서 3] (출처: /content/drive/MyDrive/Demian.pdf, 100페이지)
   내용 요약: 예상 질문들: 1. "Abraxas는 어떤 개념을 대표하며, 그것이 의미하는 바는 무엇인가요?" 2. "Demian과의 대화에서 제기된 '신성과 사탄성의 화해'에 대한 아이디어는 어떤 배경을 가지고 있나요?" 3. "주인공이 Beatrice의 형상에 대해 느끼는 변화는 무엇을 나타내고 있나요?"  원본 내용: THE BIRD STRUGGLES OUT OF ...
-------

,Question,Top_1_Context,Top_2_Context,Top_3_Context
0,싱클레어는 누구니?,"예상 질문들:\n1. 이 인물은 자신의 과거와 현재를 어떻게 대조하고 있으며, 그로...",예상 질문들:\n1. 이 그림이 왜 비아트리스의 삶에 큰 영향을 미쳤는가?\n2. ...,"예상 질문들:\n1. ""Abraxas는 어떤 개념을 대표하며, 그것이 의미하는 바는..."
1,데미안이 싱클레어에게 카인과 아벨에 대해 설명한 내용은 무엇인가요?,예상 질문들:\n1. 데미안이 자신의 의지를 통해 다른 사람에게서 원하는 것을 얻기...,"예상 질문들:\n1. ""주인공이 '진짜 데미안'이라고 느낀 이유는 무엇인가요?""\n...",예상 질문들:\n1. Sinclair와 Beck이 저녁에 무엇을 하기로 했나요?\n...
2,크로머는 어떤 방식으로 싱클레어를 협박했나요?,"예상 질문들:\n1. 어떤 요청이 Kromer에 의해 제기되었고, 그 요청에 대한 ...",예상 질문들:\n1. 이 인물의 외모에 대한 묘사는 어떤 특징을 강조하고 있나요?\...,예상 질문들:\n1. 어떤 특정한 종류의 나방에서 수컷이 암컷을 찾기 위해 얼마나 ...


### 지표 해석
Context Recall : 1

→ 이번에도 100%다. Reverse HyDE가 관련 문서를 찾아오는 능력은 확실히 좋은 것 같다.

Context Precision : 0, 0.3, 1.0

→ 질문과 관련없는 정보가 상위권에 있음을 확인할 수 있다.

Faithfulness : 0.8, 0, 1

→ LLM의 답변이 문서 내용과 관련있다고 볼 수 있다.
<br>
<br>
<br>
ReverseHyDE만 적용한 코드(rag-tutorial_reverse-hyde.ipynb)와 RAGAS를 적용한 코드(ragas-tutorial_reverse-hyde_1.ipynb)를 비교했을 때, RAGAS를 사용한 결과를 보고 실망했다. 계속 모르겠다고 말만 하다니....

코드랑 라이브러리 환경이 다소 달랐지만, 내용은 거의 비슷하다고 생각해서, 비슷한 성능을 가지고 정량 평가만 추가될 거라고 기대했었다. <br>
자세히 살펴보면 아주 조금씩 바뀐 부분이 있긴 한데, 설마 달라지겠나 싶어서 조건을 이전 코드와 똑같이 적용해서 다시 한번 코드를 돌렸다. <br>
<br>
[질문을 뽑아내는 LLM의 조건] <br>
model="gpt-4o-mini", temperature=0.7

[답변을 내는 LLM의 조건] <br>
model="gpt-4o", temperature=0.0

→ 이전 코드와 같은 조건으로 동일하게 적용<br>
<br>

☞ 결론 : 모르겠다고 대답 안한다. 그리고 대답의 내용이 맞는지는 모르나, 질문과 관련된 '대답'은 하고는 있었다.

## 회고
코드 완성하는 데에 시간이 꽤나 오래 걸렸다. 계속된 에러 속에, 제공된 예제 코드와 계속 비교해보면서 내가 뭘 잘못 작성했을까 고민을 참 많이 했던 시간이었다. 그래도 오랜 시간 쏟은 만큼 코드 읽는 버거움이 아주 조금은 사라진 기분이라 나름 뿌듯함도 느낀다.

다만 이 과제에서 핵심이었던 RAGAS를 해석하고 고치는 데에 시간을 더 쓰지 못한 게 아쉽기는 하다. 기존 실습 코드와 마지막으로 작성한 코드도 RAGAS로 비교 분석 결과를 만들고 싶었는데, 좀 아쉽다. 앞으로도 자주 볼 녀석이니까 다음에 다른 과제 제출하면서 써먹어야겠다는 생각이 든다.